<a href="https://colab.research.google.com/github/srihari-g-acharya-ci109/ML2LAB/blob/main/ML5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install networkx matplotlib pandas seaborn -q

import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# background facts for family relationships
facts = {
    "Father": {("Henry", "Tom"), ("Henry", "Sharon"), ("Henry", "Bob")},
    "Mother": {("Ann", "Tom"), ("Ann", "Sharon"), ("Ann", "Bob")},
    "Male":   {("Henry",), ("Tom",), ("Bob",)},
    "Female": {("Ann",), ("Sharon",)},
}

# training examples for target Daughter(x, y)
examples = [
    ("Sharon", "Henry", True),
    ("Bob",    "Henry", False),
    ("Tom",    "Ann",   False),
    ("Sharon", "Ann",   True),
]

candidate_literals = {
    "Father(y,x)": lambda x, y: (y, x) in facts["Father"],
    "Mother(y,x)": lambda x, y: (y, x) in facts["Mother"],
    "Male(x)":     lambda x, y: (x,) in facts["Male"],
    "Female(x)":   lambda x, y: (x,) in facts["Female"],
}

def evaluate_rule(literal_names, examples):
    pos, neg, covered = 0, 0, []
    for x, y, label in examples:
        satisfied = all(candidate_literals[lit](x, y) for lit in literal_names)
        if satisfied:
            covered.append((x, y, label))
            if label:
                pos += 1
            else:
                neg += 1
    return pos, neg, covered

def learn_one_rule(examples, candidate_literals):
    rule_body = []
    history = []
    round_num = 0
    
    while True:
        pos, neg, covered = evaluate_rule(rule_body, examples)
        history.append({
            "round": round_num,
            "rule": " AND ".join(rule_body) if rule_body else "(most general)",
            "pos_covered": pos,
            "neg_covered": neg,
        })
        if neg == 0:
            break

        scores = {}
        for lit in candidate_literals:
            if lit in rule_body:
                continue
            trial_body = rule_body + [lit]
            p, n, _ = evaluate_rule(trial_body, examples)
            scores[lit] = (p - n, p, n)

        best_lit = max(scores, key=lambda l: scores[l][0])
        rule_body.append(best_lit)
        round_num += 1

    final_pos, final_neg, final_covered = evaluate_rule(rule_body, examples)
    return rule_body, history, final_covered

rule_body, history, final_covered = learn_one_rule(examples, candidate_literals)
rule_str = " AND ".join(rule_body) + " -> Daughter(x, y)"
print("Learned rule:", rule_str)
print()
for h in history:
    print(f"Round {h['round']}: {h['rule']:35s} | pos={h['pos_covered']} neg={h['neg_covered']}")

In [ ]:
# family tree visualization
FT = nx.DiGraph()
FT.add_edges_from([
    ("Henry", "Tom"), ("Henry", "Sharon"), ("Henry", "Bob"),
    ("Ann", "Tom"), ("Ann", "Sharon"), ("Ann", "Bob"),
])

pos_ft = {
    "Henry": (-1, 1), "Ann": (1, 1),
    "Tom": (-1.5, 0), "Sharon": (0, 0), "Bob": (1.5, 0),
}

male_color, female_color = "#5da8ff", "#ff8fb0"
node_colors = [male_color if n in ("Henry", "Tom", "Bob") else female_color for n in FT.nodes()]

plt.figure(figsize=(7, 4))
nx.draw(
    FT, pos_ft, with_labels=True, node_size=2200,
    node_color=node_colors, edgecolors="#0f5f52", linewidths=2,
    font_size=10, font_weight="bold", arrows=True, arrowsize=16,
    edge_color="#0f5f52", width=2
)
plt.title("Family Tree: Background Relations", fontsize=12)
plt.axis("off")
plt.tight_layout()
plt.show()

# per-round coverage bars
rounds = [h["round"] for h in history]
pos_vals = [h["pos_covered"] for h in history]
neg_vals = [h["neg_covered"] for h in history]
x_idx = range(len(rounds))

plt.figure(figsize=(8, 4))
plt.bar([i - 0.2 for i in x_idx], pos_vals, width=0.4, label="Positives covered", color="#1a8a72")
plt.bar([i + 0.2 for i in x_idx], neg_vals, width=0.4, label="Negatives covered", color="#c0392b")
plt.xticks(list(x_idx), [f"Round {r}" for r in rounds])
plt.ylabel("Examples covered")
plt.title("Coverage per Specialization Step")
plt.legend()
plt.tight_layout()
plt.show()

# verify against training instances
verify_rows = []
for x, y, label in examples:
    lit_results = {lit: candidate_literals[lit](x, y) for lit in rule_body}
    satisfied = all(lit_results.values())
    row = {"Instance": f"({x}, {y})"}
    for lit, val in lit_results.items():
        row[lit] = "T" if val else "F"
    row["Covered"] = "Yes" if satisfied else "No"
    row["Target"] = "+" if label else "-"
    verify_rows.append(row)

verify_df = pd.DataFrame(verify_rows)
print("
Verification Table:")
print(verify_df.to_string(index=False))